In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd

# ============================================================
# 0. GEOJSON COUNTRY BOUNDARIES + UK PATCH (GB + NIR)
# ============================================================
import json
import geopandas as gpd
import pandas as pd

url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
world = gpd.read_file(url)

# Find the column containing the country name
name_col = [c for c in world.columns if "name" in c.lower() or "admin" in c.lower()][0]

def _norm(s):
    return str(s).strip().lower()

# ---- UK PATCH: replace UK with Great Britain + Northern Ireland ----
uk_fp = "NUTS_Level_1_January_2018_FEB_in_the_United_Kingdom_2022_1971092392666434121.geojson"

with open(uk_fp, "r", encoding="utf-8") as f:
    uk_json = json.load(f)

uk1 = gpd.GeoDataFrame.from_features(uk_json["features"])
uk1 = uk1.set_crs("EPSG:27700", allow_override=True).to_crs(world.crs)

# NUTS1 name column
uk_name_col = "nuts118nm" if "nuts118nm" in uk1.columns else [c for c in uk1.columns if "nm" in c.lower() or "name" in c.lower()][0]

# Geometries
ni_geom = uk1.loc[uk1[uk_name_col].astype(str).str.lower().eq("northern ireland"), "geometry"]
if ni_geom.empty:
    raise ValueError("Cannot find 'Northern Ireland' in the UK NUTS1 file.")
ni_geom = ni_geom.union_all() if hasattr(ni_geom, "union_all") else ni_geom.unary_union

gb_geom = uk1.loc[~uk1[uk_name_col].astype(str).str.lower().eq("northern ireland"), "geometry"]
gb_geom = gb_geom.union_all() if hasattr(gb_geom, "union_all") else gb_geom.unary_union

# Remove UK from world
uk_aliases = {
    "united kingdom",
    "united kingdom of great britain and northern ireland",
    "uk",
}
world = world[~world[name_col].astype(str).map(_norm).isin(uk_aliases)].copy()

# Add two new features
gb_row = {col: None for col in world.columns}
gb_row[name_col] = "Great Britain"
gb_row["geometry"] = gb_geom

ni_row = {col: None for col in world.columns}
ni_row[name_col] = "Northern Ireland"
ni_row["geometry"] = ni_geom

world = pd.concat([world, gpd.GeoDataFrame([gb_row, ni_row], crs=world.crs)], ignore_index=True)

print(f"Country name column: {name_col} — Total countries: {len(world[name_col])}")
print("UK present?", (world[name_col].astype(str).str.lower() == "united kingdom").any())
print("GB present?", (world[name_col].astype(str).str.lower() == "great britain").any())
print("NIR present?", (world[name_col].astype(str).str.lower() == "northern ireland").any())

# ============================================================
# 1. Compute MAE and Correlation between REAL and LLM by country
# ============================================================

# Load data
df = pd.read_excel("country_dataset.xlsx")

# --- FIX GB/NIR MATCHING ---
# Requires country_code (or code) and/or Country_match to be present
if "country_code" not in df.columns and "code" in df.columns:
    df = df.rename(columns={"code": "country_code"})

if "country_code" in df.columns:
    cc = df["country_code"].astype(str).str.upper().str.strip()
    # Force the correct matches
    df.loc[cc.eq("GB"), "Country_match"] = "Great Britain"
    df.loc[cc.eq("NIR"), "Country_match"] = "Northern Ireland"

# Keep only rows with non-null Country_match
df = df[df["Country_match"].notna()].copy()
df["Country_match"] = df["Country_match"].astype(str)

# Use Country_match as index -> these names must already match the GeoJSON
df = df.set_index("Country_match")

# Identify REAL:: and LLM:: columns
real_cols = [c for c in df.columns if c.startswith("REAL::")]
llm_cols  = [c for c in df.columns if c.startswith("LLM::")]

# Build REAL/LLM pairs sharing the same suffix
pairs = []
for r in real_cols:
    suffix = r.split("REAL::", 1)[1]
    l = "LLM::" + suffix
    if l in llm_cols:
        pairs.append((r, l))

print("REAL–LLM pairs found:", len(pairs))

results = []

for country in df.index:
    # Extract REAL and LLM in the same order as the pairs
    real_raw = df.loc[country, [p[0] for p in pairs]]
    llm_raw  = df.loc[country, [p[1] for p in pairs]]

    # Convert to numeric
    real_arr = pd.to_numeric(real_raw, errors="coerce").to_numpy()
    llm_arr  = pd.to_numeric(llm_raw,  errors="coerce").to_numpy()

    # Position mask: use only questions with both REAL and LLM values
    mask = ~np.isnan(real_arr) & ~np.isnan(llm_arr)

    if mask.sum() == 0:
        continue

    r = real_arr[mask]
    l = llm_arr[mask]

    diff = r - l

    mae  = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff**2))
    dist = np.linalg.norm(diff)

    # Compute correlation only when meaningful
    if mask.sum() >= 2 and np.std(r, ddof=1) > 0 and np.std(l, ddof=1) > 0:
        corr = np.corrcoef(r, l)[0, 1]
    else:
        corr = np.nan

    results.append({
        "Country": country,
        "MAE": mae,
        "RMSE": rmse,
        "Euclidean_dist": dist,
        "Correlation": corr,
        "N_variables": int(mask.sum())
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No country has at least one valid REAL+LLM pair.")

# Use Country as index
results_df = results_df.set_index("Country")
results_df = results_df.sort_values("MAE")

print("\nMAE / Correlation table:")
print(results_df.head().round(3))

# ============================================================
# 2. JOIN with the GeoDataFrame using Country_match
# ============================================================

metrics_cols = ["MAE", "Correlation"]
for col in metrics_cols:
    if col not in results_df.columns:
        raise ValueError(f"Missing column '{col}' in results_df.")

# Direct merge: world[name_col] <-> results_df.index
merged = world.merge(
    results_df[metrics_cols],
    how="left",
    left_on=name_col,
    right_index=True
)

# Remove Antarctica
merged = merged[~merged[name_col].str.contains("Antarctica", case=False, na=False)].copy()

# Web Mercator projection
merged_webmerc = merged.to_crs(epsg=3857)

# ============================================================
# 2bis. Exclude selected countries from the match
#       Niger, Tanzania, Congo, Oman, North Korea (+ variants)
# ============================================================
ban_expanded = {
    "Niger",
    "Tanzania",
    "United Republic of Tanzania",
    "Congo",
    "Republic of the Congo",
    "Democratic Republic of the Congo",
    "Oman",
    "North Korea",
    "Democratic People's Republic of Korea"
}

cols_to_null = ["MAE", "Correlation"]
for col in cols_to_null:
    if col in merged_webmerc.columns:
        merged_webmerc.loc[merged_webmerc[name_col].isin(ban_expanded), col] = np.nan

# ----------------------------
# 3. MAE map
# ----------------------------
mae_col = "MAE"
mae_vals = (
    merged_webmerc[mae_col]
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if mae_vals.empty:
    raise ValueError("The MAE column is empty. Check matches and country names.")

vmin_mae, vmax_mae = mae_vals.min(), mae_vals.max()
print(f"MAE range: {vmin_mae:.3f} – {vmax_mae:.3f}")

cmap_mae = plt.cm.OrRd

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

merged_webmerc.plot(
    column=mae_col,
    cmap=cmap_mae,
    linewidth=0.1,
    ax=ax,
    edgecolor="black",
    vmin=vmin_mae,
    vmax=vmax_mae,
    missing_kwds={
        "color": "#d3d3d3",
        "edgecolor": "black",
        "linewidth": 0.1
    }
)

ax.set_aspect("equal")
ax.axis("off")
#ax.set_title("MAE — LLM vs REAL by country", fontsize=12)

# MAE colorbar
norm_mae = plt.Normalize(vmin=vmin_mae, vmax=vmax_mae)
sm_mae = plt.cm.ScalarMappable(norm=norm_mae, cmap=cmap_mae)
sm_mae._A = []

cax_mae = inset_axes(
    ax,
    width="95%",
    height="2.5%",
    loc="lower center",
    borderpad=1
)

cbar_mae = fig.colorbar(
    sm_mae,
    cax=cax_mae,
    orientation="horizontal"
)

for spine in cbar_mae.ax.spines.values():
    spine.set_linewidth(0.3)

cbar_mae.ax.tick_params(labelsize=8)
cbar_mae.set_label("Mean Absolute Error", fontsize=9)

plt.tight_layout()
plt.savefig("map_mae_llm_vs_real.jpg", dpi=300, bbox_inches="tight")
plt.close()

print("✅ MAE map saved as 'map_mae_llm_vs_real.jpg'")

# ----------------------------
# 4. Correlation map
# ----------------------------
corr_col = "Correlation"
corr_vals = (
    merged_webmerc[corr_col]
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if corr_vals.empty:
    raise ValueError("The Correlation column is empty. Check the join with results_df and country names.")

max_abs_corr = np.max(np.abs(corr_vals))
vmin_corr, vmax_corr = -max_abs_corr, max_abs_corr
print(f"Correlation range: {vmin_corr:.3f} – {vmax_corr:.3f}")

cmap_corr = LinearSegmentedColormap.from_list(
    "blue_white_red_corr",
    ["#0000ff", "#ffffff", "#ff0000"]
)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

merged_webmerc.plot(
    column=corr_col,
    cmap=cmap_corr,
    linewidth=0.1,
    ax=ax,
    edgecolor="black",
    vmin=vmin_corr,
    vmax=vmax_corr,
    missing_kwds={
        "color": "#d3d3d3",
        "edgecolor": "black",
        "linewidth": 0.1
    }
)

ax.set_aspect("equal")
ax.axis("off")
#ax.set_title("Correlation — LLM vs REAL by country", fontsize=12)

norm_corr = plt.Normalize(vmin=vmin_corr, vmax=vmax_corr)
sm_corr = plt.cm.ScalarMappable(norm=norm_corr, cmap=cmap_corr)
sm_corr._A = []

cax_corr = inset_axes(
    ax,
    width="95%",
    height="2.5%",
    loc="lower center",
    borderpad=1
)

cbar_corr = fig.colorbar(
    sm_corr,
    cax=cax_corr,
    orientation="horizontal"
)

for spine in cbar_corr.ax.spines.values():
    spine.set_linewidth(0.3)

cbar_corr.ax.tick_params(labelsize=8)
cbar_corr.set_label("Correlation", fontsize=9)

plt.tight_layout()
plt.savefig("map_corr_llm_vs_real.jpg", dpi=300, bbox_inches="tight")
plt.close()

print("✅ Correlation map saved as 'map_corr_llm_vs_real.jpg'")

# ----------------------------
# 5. Third MAE map
#    0–4 scale, values >4 shown in black
# ----------------------------
mae_vals2 = (
    merged_webmerc[mae_col]
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if mae_vals2.empty:
    raise ValueError("The MAE column is empty. Check matches and country names.")

# Fix the scale to [0, 4]
vmin_mae2, vmax_mae2 = 0.0, 4.0
print(f"MAE range for 0–4 map with >4 in black: {vmin_mae2:.3f} – {vmax_mae2:.3f}")

# Base colormap + 'over' color = black
cmap_mae2 = plt.cm.OrRd.copy()
cmap_mae2.set_over("black")

# Normalization without clipping so values >4 are treated as "over"
norm_mae2 = plt.Normalize(vmin=vmin_mae2, vmax=vmax_mae2, clip=False)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

merged_webmerc.plot(
    column=mae_col,
    cmap=cmap_mae2,
    norm=norm_mae2,
    linewidth=0.1,
    ax=ax,
    edgecolor="black",
    missing_kwds={
        "color": "#d3d3d3",
        "edgecolor": "black",
        "linewidth": 0.1
    }
)

ax.set_aspect("equal")
ax.axis("off")
#ax.set_title("MAE — LLM vs REAL, 0–4 scale, >4 in black", fontsize=12)

# MAE colorbar with extension for values >4
sm_mae2 = plt.cm.ScalarMappable(norm=norm_mae2, cmap=cmap_mae2)
sm_mae2._A = []

cax_mae2 = inset_axes(
    ax,
    width="95%",
    height="2.5%",
    loc="lower center",
    borderpad=1
)

cbar_mae2 = fig.colorbar(
    sm_mae2,
    cax=cax_mae2,
    orientation="horizontal",
    extend="max"
)

for spine in cbar_mae2.ax.spines.values():
    spine.set_linewidth(0.3)

cbar_mae2.ax.tick_params(labelsize=8)
cbar_mae2.set_label("Mean Absolute Error", fontsize=9)

plt.tight_layout()
plt.savefig("map_mae_llm_vs_real_0_4_black_over.jpg", dpi=300, bbox_inches="tight")
plt.close()

print("✅ MAE map, 0–4 scale with >4 in black, saved as 'map_mae_llm_vs_real_0_4_black_over.jpg'")